<h1 style="text-align: center;">Release Notes Agent - Jira to Confluence</h1>
<p style="text-align: center;"><em>Powered by Google ADK & Atlassian MCP</em></p>

This agent automates release notes creation from Jira work items and publishes them to Confluence.

### How It Works:
**User**
1. Provides JQL parameter - "Affected Version"

**Agent**
1. Retrieves issues with affected_version="Jan-2026" from Jira using JQL query via Atlassian MCP
2. Extract data from issues - Name, Description, ID, Link, Project, ...
3. Generates release notes for each ticket and stores everything in tool_context
4. Publish to Confluence - Creates a formatted Atlassian Document Format (ADF) page directly via atlassian-python-api

### Architecture:
```
┌──────────────────────────────────────────────────────────────┐
│                       SequentialAgent                        │
│                    (ReleaseNotesWorkflow)                    │
├──────────────────────────────┬───────────────────────────────┤
│     TicketProcessorAgent     │   ConfluencePublisherAgent    │
│  ┌────────────────────────┐  │  ┌─────────────────────────┐  │
│  │ Atlassian MCP          │  │  │ publish_to_confluence   │  │
│  │ store_ticket_in_context│  │  │                         │  │
│  └────────────────────────┘  │  └─────────────────────────┘  │
└──────────────────────────────┴───────────────────────────────┘
```

### Local Tools:
- store_ticket_in_context: Saves each processed ticket to tool_context
- format_confluence_content: Reads tickets from context and formats as Markdown

### Prerequisites:
- Python 3.10+ with virtual environment
- Node.js 18+ - Required for MCP servers (npx)
- Atlassian Cloud Access - Browser authentication triggered on first run
- Environment Variables - GOOGLE_API_KEY in .env file

### Usage:
    await chat(affected_version="Jan-2026")


In [ ]:
%pip install google-adk mcp python-dotenv atlassian-python-api

In [2]:
import os
import json
import warnings

warnings.filterwarnings("ignore", message=".*EXPERIMENTAL.*")

from dotenv import load_dotenv
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset, StdioConnectionParams, StdioServerParameters
from google.adk.tools import ToolContext
from atlassian import Confluence

load_dotenv()

True

In [3]:
# MODEL_NAME = "gemini-3-flash-preview"
MODEL_NAME = "gemini-2.5-flash"
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# CONFLUENCE_SPACE_ID = "98305"
CONFLUENCE_SPACE_ID = "JENY"
CONFLUENCE_PARENT_PAGE_ID = "194609155"
CONFLUENCE_URL = os.getenv("JIRA_URL")
CONFLUENCE_USER = os.getenv("JIRA_EMAIL")
CONFLUENCE_TOKEN = os.getenv("JIRA_API_TOKEN")

if GOOGLE_API_KEY:
    print("✅ Google API key loaded!")
else:
    print("❌ Google API key missing!")

✅ Google API key loaded!


In [4]:
RELEASE_NOTES_INSTRUCTIONS = """
## Guidelines for Creating Release Notes

### Format
- Write in clear, concise language that end-users can understand
- Keep each release note to 2-3 sentences maximum
- Start with an action verb (Added, Fixed, Improved, Updated, etc.)

### Content Guidelines
- Focus on what is added/changed/fixed and the USER BENEFIT, not the technical implementation
- Remove internal jargon and technical details
- Highlight what was added or changed and why it matters to users

### What to Include
- New features or functionality
- Improvements to existing features
- Bug fixes (describe what was fixed)
- UI/UX changes

### What to Exclude
- Internal ticket information
- Developer-focused technical details
- Code changes or implementation specifics

### Example
TICKET DESCRIPTION:
"Implemented caching layer for user profile API calls to reduce database load.
Added Redis integration with 5-minute TTL."

RELEASE NOTE:
"Improved: User profile pages now load faster due to optimized data retrieval."
"""

In [5]:
async def store_ticket_in_context(
    tool_context: ToolContext,
    ticket_id: str,
    name: str,
    description: str,
    link: str,
    project: str,
    release_note: str,
    release_notes_summary: str
) -> dict:
    """
    Store a single processed ticket in the tool context.
    
    Call this for EACH ticket after extracting its details and generating the release note.
    
    Args:
        tool_context: The tool context for state management
        ticket_id: Jira issue key (e.g., "PROJ-123")
        name: Issue summary/title
        description: Full issue description
        link: URL to the Jira issue
        project: Jira project name
        release_note: Generated user-friendly release note (2-3 sentences)
        release_notes_summary: Short summary title (5-10 words)
    
    Returns:
        dict: Status with ticket ID confirmation
    """
    # Ensure storage is initialized
    if "processed_tickets" not in tool_context.state:
        tool_context.state["processed_tickets"] = []
        tool_context.state["tickets_count"] = 0
    
    ticket_data = {
        "id": ticket_id,
        "name": name,
        "description": description,
        "link": link,
        "project": project,
        "release_note": release_note,
        "release_notes_summary": release_notes_summary
    }
    
    tool_context.state["processed_tickets"].append(ticket_data)
    tool_context.state["tickets_count"] = len(tool_context.state["processed_tickets"])
    
    return {
        "status": "success",
        "ticket_id": ticket_id,
        "total_tickets_stored": tool_context.state["tickets_count"]
    }


async def publish_to_confluence(tool_context: ToolContext) -> dict:
    """
    Format tickets from context as ADF and publish directly to Confluence.

    Reads processed_tickets and affected_version from tool context,
    builds an ADF document, and creates a Confluence page.

    Returns:
        dict: Status and URL of the created page
    """
    tickets = tool_context.state.get("processed_tickets", [])
    affected_version = tool_context.state.get("affected_version", "Unknown")

    if not tickets:
        return {"status": "error", "message": "No tickets found in context"}

    # Build ADF table rows from tickets
    table_rows = []
    for ticket in tickets:
        table_rows.append({
            "type": "tableRow",
            "content": [
                {
                    "type": "tableCell",
                    "attrs": {"colspan": 1, "rowspan": 1},
                    "content": [{"type": "paragraph", "content": [{"text": ticket.get("project", ""), "type": "text"}]}]
                },
                {
                    "type": "tableCell",
                    "attrs": {"colspan": 1, "rowspan": 1},
                    "content": [{"type": "paragraph", "content": [{"text": ticket.get("release_notes_summary", ""), "type": "text"}]}]
                },
                {
                    "type": "tableCell",
                    "attrs": {"colspan": 1, "rowspan": 1},
                    "content": [{"type": "paragraph", "content": [{"text": ticket.get("release_note", ""), "type": "text"}]}]
                }
            ]
        })

    # Build full ADF body
    adf_body = {
        "version": 1,
        "type": "doc",
        "content": [
            {
                "type": "paragraph",
                "content": [{"text": "This release includes the following updates and improvements.", "type": "text"}]
            },
            {
                "type": "table",
                "attrs": {"layout": "default"},
                "content": [
                    {
                        "type": "tableRow",
                        "content": [
                            {
                                "type": "tableHeader",
                                "attrs": {"colspan": 1, "rowspan": 1},
                                "content": [{"type": "paragraph", "content": [{"text": "Project", "type": "text"}]}]
                            },
                            {
                                "type": "tableHeader",
                                "attrs": {"colspan": 1, "rowspan": 1},
                                "content": [{"type": "paragraph", "content": [{"text": "Summary", "type": "text"}]}]
                            },
                            {
                                "type": "tableHeader",
                                "attrs": {"colspan": 1, "rowspan": 1},
                                "content": [{"type": "paragraph", "content": [{"text": "Details", "type": "text"}]}]
                            }
                        ]
                    },
                    *table_rows
                ]
            }
        ]
    }

    page_title = f"Release Notes - {affected_version}"

    confluence = Confluence(
        url=CONFLUENCE_URL,
        username=CONFLUENCE_USER,
        password=CONFLUENCE_TOKEN,
        cloud=True
    )

    result = confluence.create_page(
        space=CONFLUENCE_SPACE_ID,
        title=page_title,
        body=json.dumps(adf_body),
        parent_id=CONFLUENCE_PARENT_PAGE_ID,
        representation="atlas_doc_format"
    )

    page_id = result.get("id")
    page_url = f"{CONFLUENCE_URL}/wiki/spaces/{CONFLUENCE_SPACE_ID}/pages/{page_id}"

    return {
        "status": "success",
        "page_title": page_title,
        "page_url": page_url
    }

In [6]:
ticket_processor_agent = LlmAgent(
    model=MODEL_NAME,
    name="TicketProcessorAgent",
    description="Fetches tickets from Jira and creates release notes",
    instruction=f"""
        You are a Release Notes Creator for Jira issues.
        
        You have the following tools:
        * searchJiraIssuesUsingJql – (Atlassian MCP tool) Search Jira issues using JQL query
        * store_ticket_in_context – (local tool) Save each processed ticket to context
        
        **STEP 1: SEARCH JIRA FOR TICKETS**
        
        Call `searchJiraIssuesUsingJql` with this JQL query:
        "Release Notes Relevant" = Yes AND affectedVersion = "{{affected_version}}"
        
        Extract all issue keys from the results.
        
        **STEP 2: PROCESS EACH TICKET (ONE BY ONE)**
        
        For EACH issue returned, do the following steps IMMEDIATELY before moving to the next:
        
        a) Extract these fields from the search results or fetch additional details if needed:
           - id: Issue key (e.g., "PROJ-123")
           - name: Issue summary/title
           - description: Issue description (full text)
           - link: URL to the issue (format: https://your-domain.atlassian.net/browse/ISSUE-KEY)
           - project: Project name or key
        
        b) Generate a release note from the description using these guidelines:
        
        {RELEASE_NOTES_INSTRUCTIONS}
        
        c) Generate a release_notes_summary: Create a short, user-friendly but meaningful title (5-10 words max) 
           that summarizes what changed. This should be suitable for publishing in release notes.
           Examples:
           - "Faster profile page loading"
           - "New dashboard filtering options"
           - "Improved error handling in exports"
        
        d) Call `store_ticket_in_context` with ALL the extracted fields AND the generated release_note.
           This saves the ticket to context IMMEDIATELY.
        
        e) Move to the next issue and repeat steps a-d.
        
        **STEP 3: OUTPUT**
        
        After processing all tickets, confirm how many tickets were processed and stored in context.
        """,
    tools=[
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params=StdioServerParameters(
                    command='npx',
                    args=[
                        '-y',
                        'mcp-remote',
                        'https://mcp.atlassian.com/v1/sse'
                    ],
                ),
                timeout=120
            ),
        ),
        store_ticket_in_context,
    ],
    output_key="tickets_processed",
)


confluence_publisher_agent = LlmAgent(
    model=MODEL_NAME,
    name="ConfluencePublisherAgent",
    description="Publishes release notes to Confluence",
    instruction="""
        You are a Confluence Publisher.
        
        You have one tool:
        * publish_to_confluence – Formats tickets from context as ADF and publishes to Confluence
        
        **STEP 1:** Call `publish_to_confluence` — it reads all tickets from context,
        builds the ADF document, and creates the Confluence page automatically.
        
        **STEP 2:** Report the URL of the created page.
        Format: [Page Title](URL)
        """,
    tools=[publish_to_confluence],
    output_key="confluence_page_url",
)


root_agent = SequentialAgent(
    name="ReleaseNotesWorkflow",
    description="Creates release notes from Jira and publishes to Confluence",
    sub_agents=[
        ticket_processor_agent,
        confluence_publisher_agent
    ],
)

In [ ]:
session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent,
    app_name="release_notes_jira_agent",
    session_service=session_service
)


async def chat(affected_version: str):
    session = await session_service.create_session(
        app_name="release_notes_jira_agent",
        user_id="user",
        session_id="release_notes_session",
        state={
            "affected_version": affected_version,
            "processed_tickets": [],
            "tickets_count": 0
        }
    )

    user_message = types.Content(
        role="user",
        parts=[types.Part(text="Create release notes and publish to Confluence")]
    )

    print("=" * 60)
    print("Release Notes Agent - Jira to Confluence")
    print("=" * 60)
    print("📝 Executing workflow...")
    print("\nAgent: ", end="", flush=True)

    async for event in runner.run_async(
        user_id="user",
        session_id="release_notes_session",
        new_message=user_message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text, end="", flush=True)

    print("\n")
    print("✅ Workflow completed!")

In [7]:
await chat(affected_version="Jan-2026")

Release Notes Agent - Jira to Confluence
📝 Executing workflow...

Agent: 

I have processed and stored 2 tickets in the context.Release Notes - Jan-2026(https://jenys.atlassian.net/wiki/spaces/JENY/pages/204308481)

✅ Workflow completed!
